In [0]:

# Cell 1 — Generate fresh daily transactions
# Every run creates a NEW batch file with today's date.
# Auto Loader will detect this new file and ingest ONLY it.
# Old files are never re-read. Data grows over time.

import json
import random
import uuid
from datetime import datetime, timezone, timedelta

SANCTIONED_NAMES = [
    "Viktor Bout", "Semion Mogilevich", "Joaquin Guzman Loera",
    "Alisher Usmanov", "Ramzan Kadyrov", "Ali Khamenei",
    "Kim Jong Un", "Robert Mugabe", "Gennady Timchenko"
]
HIGH_RISK_COUNTRIES = ["KP", "IR", "MM", "RU", "BY", "CU", "SY", "YE"]
NORMAL_COUNTRIES    = ["US", "GB", "DE", "FR", "JP", "CA", "AU", "SG", "NL", "CH", "IN", "BR"]
CURRENCIES          = ["USD", "EUR", "GBP", "JPY", "CHF", "CAD", "AUD", "SGD"]
PURPOSE_CODES       = ["SALA", "SUPP", "TRAD", "LOAN", "INVS", "GDDS", "SVCS"]
NORMAL_NAMES        = [
    "Alice Johnson", "James Smith", "Maria Santos", "Wei Zhang",
    "Priya Patel", "Carlos Rivera", "Emma Wilson", "Liam Brown",
    "Yuki Tanaka", "Fatima Al-Hassan", "David Okonkwo", "Sophie Mueller",
    "Raj Sharma", "Ana Oliveira", "Chen Wei", "Mohammed Al-Rashid",
    "Isabella Ferrari", "Hiroshi Nakamura", "Amara Diallo", "Lucas Petit"
]

def make_transaction(i):
    is_suspicious = (i % 20 == 0)
    txn_type = random.choice(["sanctioned", "missing_fields", "large_amount"]) if is_suspicious else "normal"
    txn_time = datetime.now(timezone.utc) - timedelta(minutes=random.randint(0, 1440))
    return {
        "transaction_id":    str(uuid.uuid4()),
        "timestamp":         txn_time.isoformat(),
        "message_type":      "pacs.008",
        "batch_number":      int(datetime.now().strftime("%Y%m%d")),
        "sender_name":       random.choice(SANCTIONED_NAMES) if txn_type == "sanctioned" else random.choice(NORMAL_NAMES),
        "sender_account":    f"DE{''.join([str(random.randint(0,9)) for _ in range(18)])}",
        "sender_country":    random.choice(HIGH_RISK_COUNTRIES) if txn_type == "large_amount" else random.choice(NORMAL_COUNTRIES),
        "sender_address":    "" if txn_type == "missing_fields" else f"{random.randint(1,999)} Main St, City",
        "receiver_name":     random.choice(NORMAL_NAMES),
        "receiver_account":  f"GB{''.join([str(random.randint(0,9)) for _ in range(18)])}",
        "receiver_country":  random.choice(NORMAL_COUNTRIES),
        "amount":            round(random.uniform(500_000, 5_000_000), 2) if txn_type == "large_amount" else round(random.uniform(50, 10_000), 2),
        "currency":          random.choice(CURRENCIES),
        "purpose_code":      random.choice(PURPOSE_CODES),
        "transaction_type":  txn_type,
    }

# Generate today's batch
VOLUME_PATH = "/Volumes/aml_pipeline/transactions/raw_data"
BATCH_SIZE  = 10_000
TODAY       = datetime.now().strftime("%Y%m%d")
FILE_NAME   = f"batch_{TODAY}.json"
FILE_PATH   = f"{VOLUME_PATH}/{FILE_NAME}"

# Check if today's batch already exists
try:
    dbutils.fs.ls(FILE_PATH)
    print(f"Today's batch already exists: {FILE_NAME}")
    print("Skipping generation — pipeline will process any unprocessed files")
except:
    print(f"Generating {BATCH_SIZE:,} fresh transactions for {TODAY}...")
    transactions = [make_transaction(i) for i in range(1, BATCH_SIZE + 1)]
    flagged = sum(1 for t in transactions if t["transaction_type"] != "normal")

    with open(FILE_PATH, "w") as f:
        for txn in transactions:
            f.write(json.dumps(txn) + "\n")

    print(f"New batch written: {FILE_NAME}")
    print(f"  Records    : {BATCH_SIZE:,}")
    print(f"  Suspicious : {flagged}")

# Show all batch files in the volume
print(f"\nAll batch files:")
files = dbutils.fs.ls(VOLUME_PATH)
json_files = sorted([f.name for f in files if f.name.endswith('.json')])
for f in json_files:
    print(f"  {f}")
print(f"Total files: {len(json_files)}")

Generating 10,000 fresh transactions for 20260629...
New batch written: batch_20260629.json
  Records    : 10,000
  Suspicious : 500

All batch files:
  batch_002.json
  batch_003.json
  batch_004.json
  batch_005.json
  batch_006.json
  batch_007.json
  batch_008.json
  batch_009.json
  batch_010.json
  batch_011.json
  batch_20260629.json
Total files: 11


In [0]:

# Cell 2 — Real-world pipeline: Bronze append → Silver rebuild → Gold rebuild
# Bronze: APPEND ONLY — Auto Loader reads only new files, never re-reads old ones
# Silver: OVERWRITE — rebuilt from full Bronze each run (enrichment layer)
# Gold: OVERWRITE — rebuilt from full Silver flagged (business layer)
# Tables are NEVER dropped. Data grows over time.

import requests
from pyspark.sql.functions import col, udf, when, lit, current_timestamp, concat, date_format
from pyspark.sql.types import DoubleType, StringType, StructType, StructField, IntegerType

RAW_PATH        = "/Volumes/aml_pipeline/transactions/raw_data/"
BRONZE_TABLE    = "aml_pipeline.transactions.bronze_transactions"
SILVER_TABLE    = "aml_pipeline.transactions.silver_transactions"
GOLD_TABLE      = "aml_pipeline.transactions.gold_sar_reports"

# PERSISTENT checkpoint — never cleared, Auto Loader uses it to
# track which files have already been processed
CHECKPOINT_PATH = "/Volumes/aml_pipeline/transactions/raw_data/_checkpoints/bronze_persistent"

# ── BRONZE (append only) ─────────────────────────────────────
print("STEP 1: Bronze ingestion (append only — new files only)...")

schema = StructType([
    StructField("transaction_id",   StringType(),  True),
    StructField("timestamp",        StringType(),  True),
    StructField("message_type",     StringType(),  True),
    StructField("batch_number",     IntegerType(), True),
    StructField("sender_name",      StringType(),  True),
    StructField("sender_account",   StringType(),  True),
    StructField("sender_country",   StringType(),  True),
    StructField("sender_address",   StringType(),  True),
    StructField("receiver_name",    StringType(),  True),
    StructField("receiver_account", StringType(),  True),
    StructField("receiver_country", StringType(),  True),
    StructField("amount",           DoubleType(),  True),
    StructField("currency",         StringType(),  True),
    StructField("purpose_code",     StringType(),  True),
    StructField("transaction_type", StringType(),  True),
])

(
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "json")
         .option("cloudFiles.schemaLocation", CHECKPOINT_PATH + "/schema")
         .schema(schema)
         .load(RAW_PATH)
         .withColumn("ingestion_timestamp", current_timestamp())
         .withColumn("source_file",         col("_metadata.file_path"))
         .withColumn("pipeline_layer",      lit("bronze"))
         .writeStream
         .format("delta")
         .outputMode("append")
         .option("checkpointLocation", CHECKPOINT_PATH)
         .option("mergeSchema", "true")
         .trigger(availableNow=True)
         .toTable(BRONZE_TABLE)
).awaitTermination()

bronze_count = spark.table(BRONZE_TABLE).count()
print(f"Bronze total: {bronze_count:,} records (cumulative)")

# ── SILVER (full rebuild from Bronze) ────────────────────────
print("\nSTEP 2: Silver enrichment (full rebuild)...")

try:
    rates = requests.get("https://api.frankfurter.app/latest?from=USD", timeout=10).json().get("rates", {})
    rates["USD"] = 1.0
    print(f"Exchange rates fetched ({len(rates)} currencies)")
except:
    rates = {"USD":1.0,"EUR":0.92,"GBP":0.79,"JPY":149.5,"CHF":0.89,"CAD":1.36,"AUD":1.53,"SGD":1.34}
    print("Using fallback rates")

try:
    lines = requests.get("https://www.treasury.gov/ofac/downloads/sdn.csv", timeout=15).text.split("\n")
    OFAC  = set()
    for line in lines[:500]:
        parts = line.split(",")
        if len(parts) > 1:
            n = parts[1].strip().strip('"').lower()
            if n:
                OFAC.add(n)
    print(f"OFAC list loaded ({len(OFAC)} names)")
except:
    OFAC = {"viktor bout","semion mogilevich","ramzan kadyrov","ali khamenei","kim jong un"}
    print("Using fallback OFAC list")

HIGH_RISK           = {"KP","IR","MM","RU","BY","CU","SY","YE"}
LARGE_TXN_THRESHOLD = 500_000

def to_usd(amount, currency):
    if not amount or not currency:
        return None
    return round(float(amount) / float(rates.get(currency, 1.0)), 2)

def check_ofac(name):
    if not name:
        return "CLEAN"
    if name.lower().strip() in OFAC:
        return "SANCTIONS_HIT"
    return "CLEAN"

def check_travel_rule(address, sender, s_acct, receiver, r_acct):
    missing = [f for f, v in [
        ("sender_address",   address),
        ("sender_name",      sender),
        ("sender_account",   s_acct),
        ("receiver_name",    receiver),
        ("receiver_account", r_acct)
    ] if not v or not v.strip()]
    return f"TRAVEL_RULE_VIOLATION: missing {', '.join(missing)}" if missing else "COMPLIANT"

udf_usd  = udf(to_usd, DoubleType())
udf_ofac = udf(check_ofac, StringType())
udf_tr   = udf(check_travel_rule, StringType())

(
    spark.table(BRONZE_TABLE)
    .withColumn("amount_usd",                udf_usd(col("amount"), col("currency")))
    .withColumn("sender_sanctions_status",   udf_ofac(col("sender_name")))
    .withColumn("receiver_sanctions_status", udf_ofac(col("receiver_name")))
    .withColumn("travel_rule_status",        udf_tr(col("sender_address"), col("sender_name"),
                                             col("sender_account"), col("receiver_name"),
                                             col("receiver_account")))
    .withColumn("high_risk_country",         col("sender_country").isin(list(HIGH_RISK)))
    .withColumn("large_transaction",         col("amount_usd") > LARGE_TXN_THRESHOLD)
    .withColumn("is_flagged",
        (col("sender_sanctions_status") == "SANCTIONS_HIT") |
        (col("receiver_sanctions_status") == "SANCTIONS_HIT") |
        (col("travel_rule_status") != "COMPLIANT") |
        (col("high_risk_country") == True) |
        (col("large_transaction") == True))
    .withColumn("flag_reason",
        when(col("sender_sanctions_status") == "SANCTIONS_HIT", lit("OFAC: Sanctioned sender"))
        .when(col("receiver_sanctions_status") == "SANCTIONS_HIT", lit("OFAC: Sanctioned receiver"))
        .when(col("travel_rule_status") != "COMPLIANT", col("travel_rule_status"))
        .when(col("high_risk_country") == True, lit("HIGH_RISK_COUNTRY"))
        .when(col("large_transaction") == True, lit("LARGE_TRANSACTION_>500K_USD"))
        .otherwise(lit("NONE")))
    .withColumn("silver_timestamp", current_timestamp())
    .withColumn("pipeline_layer",   lit("silver"))
    .write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)
)

silver_count  = spark.table(SILVER_TABLE).count()
flagged_count = spark.table(SILVER_TABLE).filter("is_flagged = true").count()
print(f"Silver total: {silver_count:,} records")
print(f"  Flagged   : {flagged_count:,} ({round(flagged_count/silver_count*100,1)}%)")

# ── GOLD (full rebuild from Silver flagged) ──────────────────
print("\nSTEP 3: Gold SAR reports (full rebuild)...")

(
    spark.table(SILVER_TABLE)
    .filter(col("is_flagged") == True)
    .withColumn("sar_severity",
        when((col("sender_sanctions_status") == "SANCTIONS_HIT") |
             (col("receiver_sanctions_status") == "SANCTIONS_HIT"), lit("CRITICAL"))
        .when(col("high_risk_country") == True, lit("HIGH"))
        .when(col("travel_rule_status") != "COMPLIANT", lit("HIGH"))
        .otherwise(lit("MEDIUM")))
    .withColumn("sar_reference",
        concat(lit("SAR-"), date_format(current_timestamp(), "yyyyMMdd"),
               lit("-"), col("transaction_id").substr(1, 8)))
    .withColumn("report_status",         lit("PENDING_REVIEW"))
    .withColumn("reporting_institution", lit("SentinelFlow Demo Bank"))
    .withColumn("filing_deadline",       date_format(current_timestamp(), "yyyy-MM-dd"))
    .withColumn("gold_timestamp",        current_timestamp())
    .withColumn("pipeline_layer",        lit("gold"))
    .write.format("delta").mode("overwrite").saveAsTable(GOLD_TABLE)
)

gold_count = spark.table(GOLD_TABLE).count()
critical   = spark.table(GOLD_TABLE).filter("sar_severity = 'CRITICAL'").count()
high       = spark.table(GOLD_TABLE).filter("sar_severity = 'HIGH'").count()
medium     = spark.table(GOLD_TABLE).filter("sar_severity = 'MEDIUM'").count()

print(f"\n{'='*50}")
print(f"  SENTINELFLOW PIPELINE COMPLETE")
print(f"{'='*50}")
print(f"  Bronze (cumulative) : {bronze_count:,}")
print(f"  Silver              : {silver_count:,}")
print(f"  Gold (SAR reports)  : {gold_count:,}")
print(f"  Flag rate           : {round(gold_count/silver_count*100,1)}%")
print(f"  CRITICAL : {critical:,}")
print(f"  HIGH     : {high:,}")
print(f"  MEDIUM   : {medium:,}")
print(f"\n  Flag reason breakdown:")
spark.table(GOLD_TABLE).groupBy("flag_reason").count().orderBy("count", ascending=False).show(truncate=False)

STEP 1: Bronze ingestion (append only — new files only)...
Bronze total: 110,000 records (cumulative)

STEP 2: Silver enrichment (full rebuild)...
Exchange rates fetched (30 currencies)
OFAC list loaded (468 names)
Silver total: 110,000 records
  Flagged   : 3,696 (3.4%)

STEP 3: Gold SAR reports (full rebuild)...

  SENTINELFLOW PIPELINE COMPLETE
  Bronze (cumulative) : 110,000
  Silver              : 110,000
  Gold (SAR reports)  : 3,696
  Flag rate           : 3.4%
  CRITICAL : 0
  HIGH     : 3,696
  MEDIUM   : 0

  Flag reason breakdown:
+---------------------------------------------+-----+
|flag_reason                                  |count|
+---------------------------------------------+-----+
|TRAVEL_RULE_VIOLATION: missing sender_address|1859 |
|HIGH_RISK_COUNTRY                            |1837 |
+---------------------------------------------+-----+



In [0]:

# Cell 3 — Verify pipeline and show data growth over time
# This proves the pipeline is accumulating data like a real system

print("Pipeline Verification")
print("=" * 50)

b = spark.table("aml_pipeline.transactions.bronze_transactions").count()
s = spark.table("aml_pipeline.transactions.silver_transactions").count()
g = spark.table("aml_pipeline.transactions.gold_sar_reports").count()

print(f"  Bronze : {b:,} (cumulative — grows each run)")
print(f"  Silver : {s:,}")
print(f"  Gold   : {g:,}")
print(f"  Flag % : {round(g/s*100,1)}%")

# Show records per batch — proves data is coming from multiple runs
print(f"\nRecords per batch (shows pipeline growth):")
spark.table("aml_pipeline.transactions.bronze_transactions") \
    .groupBy("batch_number") \
    .count() \
    .orderBy("batch_number") \
    .show(20, truncate=False)

# Show batch files in volume
print("Batch files in volume:")
files = dbutils.fs.ls("/Volumes/aml_pipeline/transactions/raw_data/")
for f in sorted([f.name for f in files if f.name.endswith('.json')]):
    print(f"  {f}")

Pipeline Verification
  Bronze : 110,000 (cumulative — grows each run)
  Silver : 110,000
  Gold   : 3,696
  Flag % : 3.4%

Records per batch (shows pipeline growth):
+------------+-----+
|batch_number|count|
+------------+-----+
|2           |10000|
|3           |10000|
|4           |10000|
|5           |10000|
|6           |10000|
|7           |10000|
|8           |10000|
|9           |10000|
|10          |10000|
|11          |10000|
|20260629    |10000|
+------------+-----+

Batch files in volume:
  batch_002.json
  batch_003.json
  batch_004.json
  batch_005.json
  batch_006.json
  batch_007.json
  batch_008.json
  batch_009.json
  batch_010.json
  batch_011.json
  batch_20260629.json
